# 12 · Final benchmark, error analysis, scenarios
Combines all model results, compares to baselines with uncertainty, relates performance to the noise ceiling, and analyses *where* models fail. Also covers Scenario 1 (seen perturbations/unseen cells = the noise ceiling) and Scenario 3 (unseen combinations, Norman).

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import glob, seaborn as sns
from perturbai.evaluation.benchmark import summarize, compare_to_baseline
parts = [pd.read_csv(f) for f in sorted(glob.glob(str(P.results / f"{TAG}_bench_*.csv")))]
long = pd.concat(parts, ignore_index=True).drop_duplicates(["model", "seed", "perturbation"])
long.to_csv(P.results / artifact("benchmark", TAG), index=False)
S = summarize(long); display(S.round(3))
display(compare_to_baseline(long, "mean", "pearson_delta").round(3))
display(compare_to_baseline(long, "mean", "mse_delta").round(4))   # for mse, NEGATIVE diff = better

In [ ]:
ceil = pd.read_csv(P.results / f"{TAG}_noise_ceiling.csv", index_col=0)
print("Noise ceiling (split-half Pearson on delta), mean:", ceil.pearson_split_half.mean().round(3))
fig, ax = plt.subplots(figsize=(7, 3.8))
order = S.index
sns.barplot(data=long.groupby(["model", "seed"]).pearson_delta.mean().reset_index(), x="model", y="pearson_delta", order=order, errorbar="sd", ax=ax)
ax.axhline(ceil.pearson_split_half.mean(), ls="--", c="k", label="noise ceiling"); ax.legend(); ax.set_title("Unseen perturbations: Pearson on delta (mean ± sd over seeds)")
plt.tight_layout(); plt.savefig(P.figures / f"{TAG}_benchmark.png", dpi=150); plt.show()

## Error analysis: when does the best model fail?

In [ ]:
best = S.index[0]; print("best model:", best)
n_cells = pd.read_csv(P.processed / artifact("n_cells", TAG), index_col=0).iloc[:, 0]
delta = pd.read_parquet(P.processed / artifact("effects", TAG))
b = long[long.model == best].groupby("perturbation").pearson_delta.mean().to_frame("pearson")
b["l2_effect"] = np.linalg.norm(delta.loc[b.index].values, axis=1); b["n_cells"] = n_cells.reindex(b.index)
b["ceiling"] = ceil.pearson_split_half.reindex(b.index)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, c in zip(ax, ["l2_effect", "n_cells", "ceiling"]):
    a.scatter(b[c], b.pearson, s=10); a.set(xlabel=c, ylabel="pearson (delta)")
plt.tight_layout(); plt.show()
print(b.corr(method="spearman").round(2))
print("Best / worst predicted perturbations:"); display(b.sort_values("pearson").iloc[[0, 1, 2, -3, -2, -1]])

## Scenario 3 — unseen combinations (Norman 2019, CRISPRa) — optional
Separate experiment; **not** mixed with Replogle. Baseline: additive model delta(A+B) = delta(A)+delta(B). A learned model must beat additivity to be interesting (Norman combos contain genuine genetic interactions).

In [ ]:
RUN_NORMAN = False
if RUN_NORMAN:
    from perturbai.data.loaders import load_dataset
    from perturbai.preprocessing.qc import normalize_log1p_matrix
    from perturbai.features.effects import compute_effects
    from perturbai.models.baselines import predict_additive
    from perturbai.evaluation.metrics import per_perturbation_metrics
    nd = CFG["datasets"]["norman2019"]; SEP = nd["combo_separator"]
    na = load_dataset("norman2019", P)
    lab = na.obs[nd["perturbation_col"]].astype(str).values
    _, Xl = normalize_log1p_matrix(na.X)
    res = compute_effects(Xl, lab, na.var_names, nd["control_label"], min_cells=20)
    D = res.delta
    # NOTE: check how combos are named in your file (e.g. 'AHR_KLF1' or 'AHR+KLF1'); adjust SEP accordingly
    singles = [p for p in D.index if SEP not in p and "+" not in p]; combos = [p for p in D.index if SEP in p]
    pred = predict_additive(D.loc[singles], [c.replace("+", SEP) for c in combos], sep=SEP)
    obs_ = D.loc[[c for c in combos if c.replace("+", SEP) in pred.index]]
    m = per_perturbation_metrics(pred.loc[[c.replace("+", SEP) for c in obs_.index]].values, obs_.values, res.ctrl_mean.values, index=obs_.index)
    print(m[["pearson_delta", "mse_delta", "direction_acc_top_deg"]].describe().round(3))

## Save the best model for the app
Refit on **all** measured perturbations and store everything the app/API needs.

In [ ]:
import joblib
from perturbai.pipeline import load_task
from perturbai.models.baselines import RidgeModel, KNNModel, RandomForestModel
from perturbai.models.mlp import MLPModel
T = load_task(P, TAG, ("coexp_",)); Mc = MCFG["models"]
cands = {"ridge": lambda: RidgeModel(**Mc["ridge"]), "knn": lambda: KNNModel(**Mc["knn"]),
         "rf": lambda: RandomForestModel(Mc["rf"]["n_estimators"], Mc["rf"]["n_pca_targets"], Mc["rf"]["min_samples_leaf"]),
         "mlp": lambda: MLPModel(hidden=Mc["mlp"]["hidden"], dropout=Mc["mlp"]["dropout"], epochs=Mc["mlp"]["epochs"] if not SYNTHETIC else 80)}
usable = [m for m in S.index if m in cands]
final_name = usable[0]; print("final model:", final_name)
model = cands[final_name]().fit(T["X"], T["Y"])
if hasattr(model, "net"): model.net = model.net.cpu(); model.dev = "cpu"
joblib.dump({"name": final_name, "model": model, "feature_table": T["feats"], "genes": list(T["genes"]),
             "ctrl_mean": T["ctrl_mean"], "trained_on": list(T["index"]), "tag": TAG}, P.models / artifact("best_model", TAG))
print("saved ->", P.models / artifact("best_model", TAG))

## Conclusions template
1. Does any model beat the mean baseline on unseen perturbations? By how much, relative to seed variation?
2. How close is the best model to the noise ceiling?
3. Which perturbations are predictable (strong effects, many cells, gene features informative)?
4. Is deep learning justified here? State your evidence.